# 05. Weighted Surrogate (가중 대리모형)

Tree depth=1 기준으로, 거절 대상(하위 점수)에 가중치를 부여하여 학습한다.
거절자 비율: 5%, 10%, 20%, 30%, 40%, 50%

가중치 alpha=2: 거절 대상 샘플에 2배 가중.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import TreeSurrogate

DATA_DIR = '../.data'

In [2]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(1 - bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(1 - bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(1 - bm.predict_proba(s['X_test'])[:, 1]),
    }

In [3]:
def advfull_metrics(bb_adv_list, surr_adv_list):
    """AdvFull: Recall & Jaccard of adverse feature sets (feature name based).
    BB adverse: contrib > 0 in log-odds (감점).
    Surr adverse: contrib < 0 in score space (감점).
    """
    recalls, jaccards = [], []
    for i in range(len(bb_adv_list)):
        bb_set = set(bb_adv_list[i])
        su_set = set(surr_adv_list[i])
        if len(bb_set) == 0:
            continue
        inter = len(bb_set & su_set)
        recalls.append(inter / len(bb_set))
        union = len(bb_set | su_set)
        jaccards.append(inter / union if union > 0 else 0.0)
    r = round(float(np.mean(recalls)), 4) if recalls else 0.0
    j = round(float(np.mean(jaccards)), 4) if jaccards else 0.0
    return r, j


def advfull_from_contribs(contribs, bb_adv_list, feature_names):
    """AdvFull from raw contribs array (for calibrated results)."""
    surr_adv = []
    for i in range(len(contribs)):
        neg_idx = np.where(contribs[i] < 0)[0]
        if len(neg_idx) == 0:
            surr_adv.append(np.array([], dtype='<U1'))
        else:
            neg_order = neg_idx[np.argsort(contribs[i][neg_idx])]
            surr_adv.append(feature_names[neg_order])
    return advfull_metrics(bb_adv_list, surr_adv)


# BB ground truth
def get_bb(data_flag):
    bb_contrib = base_models[data_flag]['bb_contribs']['lgb']
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)
    order = np.argsort(np.abs(bb_contrib), axis=1)[:, ::-1]
    bb_rank = feature_names[order]
    bb_adv = []
    for i in range(len(bb_contrib)):
        pos_idx = np.where(bb_contrib[i] > 0)[0]
        if len(pos_idx) == 0:
            bb_adv.append(np.array([], dtype='<U1'))
        else:
            pos_order = pos_idx[np.argsort(bb_contrib[i][pos_idx])[::-1]]
            bb_adv.append(feature_names[pos_order])
    return bb_rank, bb_adv

def evaluate_surrogate(surr, X, y, bb_rank=None, bb_adv=None, k=4):
    pred = surr.predict(X)
    result = {'R2': round(r2_score(y, pred), 4),
              'Spearman': round(float(spearmanr(y, pred)[0]), 4)}
    if bb_rank is not None and hasattr(surr, 'contribution_ranking'):
        surr_rank = surr.contribution_ranking(X)
        result[f'Top{k}'] = round(float(np.mean([
            len(set(bb_rank[i,:k]) & set(surr_rank[i,:k]))/k for i in range(len(X))])), 4)
    if bb_adv is not None and hasattr(surr, 'adverse_features'):
        surr_adv = surr.adverse_features(X)
        adv_ov = []
        for i in range(len(X)):
            if len(bb_adv[i])==0 or len(surr_adv[i])==0: continue
            ke = min(k, len(bb_adv[i]), len(surr_adv[i]))
            adv_ov.append(len(set(bb_adv[i][:ke]) & set(surr_adv[i][:ke]))/ke)
        result[f'AdvTop{k}'] = round(float(np.mean(adv_ov)), 4) if adv_ov else 0.0

    if bb_adv is not None and hasattr(surr, 'adverse_features'):
        surr_adv = surr.adverse_features(X)
        recall, jaccard = advfull_metrics(bb_adv, surr_adv)
        result['AdvFull_R'] = recall
        result['AdvFull_J'] = jaccard
    return result


In [4]:
REJECT_PCTS = [5, 10, 20, 30, 40, 50]
ALPHA = 2.0

for data_flag in ['GMSC', 'HC']:
    s = base_models[data_flag]['splits']
    t = targets[data_flag]
    X_tr, X_val, X_test = s['X_tr'], s['X_val'], s['X_test']
    y_tr, y_val, y_test = t['y_logit_tr'], t['y_logit_val'], t['y_logit_test']
    bb_rank, bb_adv = get_bb(data_flag)

    print(f'\n{"#"*80}')
    print(f'  {data_flag}')
    print(f'{"#"*80}')

    # Baseline (no weight)
    surr_base = TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0)
    surr_base.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    print(f'\n  Baseline (no weight):')
    print(f'    full:   {evaluate_surrogate(surr_base, X_test, y_test, bb_rank, bb_adv)}')

    for pct in REJECT_PCTS:
        # 하위 pct% = 점수가 낮은 쪽 = 거절 대상
        cutoff = np.percentile(y_tr, pct)
        reject_tr = y_tr <= cutoff

        # 가중치: 거절 대상에 alpha배
        w = np.ones(len(y_tr))
        w[reject_tr] = ALPHA

        surr_w = TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0)
        surr_w.fit(X_tr, y_tr, eval_set=(X_val, y_val), sample_weight=w)

        # 전체 평가
        res_full = evaluate_surrogate(surr_w, X_test, y_test, bb_rank, bb_adv)

        # 거절자 한정 평가
        cutoff_test = np.percentile(y_test, pct)
        reject_test = y_test <= cutoff_test
        res_reject = evaluate_surrogate(
            surr_w, X_test[reject_test], y_test[reject_test],
            bb_rank[reject_test], [bb_adv[i] for i in range(len(bb_adv)) if reject_test[i]])

        print(f'\n  reject={pct:>2d}% (alpha={ALPHA}):')
        print(f'    full:   {res_full}')
        print(f'    reject: {res_reject}')


################################################################################
  GMSC
################################################################################



  Baseline (no weight):


    full:   {'R2': 0.9332, 'Spearman': 0.969, 'Top4': 0.8467, 'AdvTop4': 0.8988, 'AdvFull_R': 0.8366, 'AdvFull_J': 0.7583}



  reject= 5% (alpha=2.0):
    full:   {'R2': 0.9325, 'Spearman': 0.9678, 'Top4': 0.8457, 'AdvTop4': 0.9043, 'AdvFull_R': 0.8189, 'AdvFull_J': 0.7735}
    reject: {'R2': -1.0326, 'Spearman': 0.6853, 'Top4': 0.8873, 'AdvTop4': 0.9369, 'AdvFull_R': 0.8764, 'AdvFull_J': 0.8643}



  reject=10% (alpha=2.0):
    full:   {'R2': 0.9323, 'Spearman': 0.9675, 'Top4': 0.8458, 'AdvTop4': 0.9025, 'AdvFull_R': 0.8211, 'AdvFull_J': 0.7713}
    reject: {'R2': 0.1452, 'Spearman': 0.8239, 'Top4': 0.8463, 'AdvTop4': 0.916, 'AdvFull_R': 0.8629, 'AdvFull_J': 0.845}


...


  reject=10% (alpha=2.0):
    full:   {'R2': 0.8459, 'Spearman': 0.9214, 'Top4': 0.5964, 'AdvTop4': 0.6119, 'AdvFull_R': 0.6883, 'AdvFull_J': 0.5959}
    reject: {'R2': -0.3677, 'Spearman': 0.5697, 'Top4': 0.5831, 'AdvTop4': 0.6178, 'AdvFull_R': 0.7256, 'AdvFull_J': 0.6512}



  reject=20% (alpha=2.0):
    full:   {'R2': 0.8437, 'Spearman': 0.9208, 'Top4': 0.5975, 'AdvTop4': 0.6122, 'AdvFull_R': 0.6914, 'AdvFull_J': 0.5984}
    reject: {'R2': 0.2139, 'Spearman': 0.6643, 'Top4': 0.5814, 'AdvTop4': 0.6214, 'AdvFull_R': 0.7244, 'AdvFull_J': 0.6444}



  reject=30% (alpha=2.0):
    full:   {'R2': 0.8434, 'Spearman': 0.9209, 'Top4': 0.5976, 'AdvTop4': 0.6084, 'AdvFull_R': 0.69, 'AdvFull_J': 0.5984}
    reject: {'R2': 0.4453, 'Spearman': 0.7246, 'Top4': 0.5787, 'AdvTop4': 0.6216, 'AdvFull_R': 0.7188, 'AdvFull_J': 0.6374}



  reject=40% (alpha=2.0):
    full:   {'R2': 0.8438, 'Spearman': 0.9213, 'Top4': 0.5995, 'AdvTop4': 0.6089, 'AdvFull_R': 0.6869, 'AdvFull_J': 0.5959}
    reject: {'R2': 0.5715, 'Spearman': 0.7691, 'Top4': 0.5799, 'AdvTop4': 0.6207, 'AdvFull_R': 0.7121, 'AdvFull_J': 0.6294}



  reject=50% (alpha=2.0):
    full:   {'R2': 0.8443, 'Spearman': 0.9216, 'Top4': 0.6013, 'AdvTop4': 0.6093, 'AdvFull_R': 0.6901, 'AdvFull_J': 0.5985}
    reject: {'R2': 0.6527, 'Spearman': 0.8034, 'Top4': 0.5808, 'AdvTop4': 0.6218, 'AdvFull_R': 0.7117, 'AdvFull_J': 0.6267}
